In [0]:
%pip install  /dbfs/FileStore/shared_uploads/dmat/dmat-0.1-py3-none-any.whl

In [0]:
dbutils.widgets.text("process_start_date", "2025-10-17", "Start Date")
dbutils.widgets.text("process_end_date", "2025-10-25", "End Date")

#remove widgets
#dbutils.widgets.removeAll()

In [0]:
%scala
// Imports & Inits
import com.poshmark.spark.helpers._
import org.apache.spark.sql._
import org.apache.spark._
import sqlContext.implicits._
import org.joda.time._
import org.joda.time.format.DateTimeFormat
import org.apache.spark.storage.StorageLevel
import org.joda.time.format.DateTimeFormat
import org.apache.spark.sql.SaveMode

// Load default configs for helper libraries and register UDFs
Config.reloadConfigs(sc)
Config.registerUDFs()

// Get create a localDate and get yesterday's date
val localDate = new LocalDate(DateTimeZone.forID( "US/Pacific" ))
val process_start_date1 = localDate.minusDays(1).toString()
val process_end_date1 = localDate.minusDays(3).toString()

// Helper method for handling mepty strings
def isEmpty(x: String) = x == null || x.trim.isEmpty

// How to get a value from a widget parameter
var process_start_date = dbutils.widgets.get("process_start_date")
var process_end_date = dbutils.widgets.get("process_end_date")
//var process_start_date = ("2025-07-31")
//var process_end_date = ("2025-07-01")
// Handle conditional values. If widget is empty set the date to yesterday
if (isEmpty(process_start_date)) {process_start_date = process_start_date1}
if (isEmpty(process_end_date)) {process_end_date = process_end_date1}
// Add additional spark configurations as needed
// Documentation:
// The below code changes overwrite strategy with partitions. 
// Instead of deleting all data in a path it will only delete data in the partitions we're writing to.
spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")

////////////////////////////
//// Read from Redshift (if data doesn't exist in Hive/S3 already)
//Redshift.getQueryDF("select 1 as test").createOrReplaceTempView("redshift_data")
//var rdf = spark.sql("select * from redshift_data")

////////////////////////////
//// Read from Hive (tables on top of s3 files)
//var sdf = spark.sql("""
// select *
// from s3_analytics.dw_users
// limit 5
// """)
//sdf.createOrReplaceTempView("dw_users")

////////////////////////////
//// Read from S3 directly via spark dataframe (table creation can be skipped)
//var sdf = spark.read.parquet("s3://data-tmp-poshmark-production/dmat/project_highway/main_page/")
//sdf.createOrReplaceTempView("main_page_list")

////////////////////////////
//// Read from S3 directly via sql (table creation can be skipped)
//var sdf = spark.sql("""
// select * 
// from parquet.`s3://data-tmp-poshmark-production/dmat/project_highway/main_page/`
// """)
//sdf.createOrReplaceTempView("main_page_list")


////////////////////////////
//// Write to S3 - Fail if data already exists (default)
//sdf.write.format("parquet").save("s3://data-tmp-poshmark-production/steve/examples/test/")

////////////////////////////
//// Write to S3 - Overwrite existing data
//sdf.write.format("parquet").mode("Overwrite").save("s3://data-tmp-poshmark-production/steve/examples/test/")

////////////////////////////
//// Write to S3 - Partition & overwrite existing data in the partitions we're writing
//spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")
//sdf.write.partitionBy("event_date").format("parquet").mode("Overwrite").save("s3://data-tmp-poshmark-production/steve/examples/test/")

////////////////////////////
//// Write to Redshift
//var table_name = "analytics_scratch.tmp_test_table"
//Redshift.writeDF(
//     df=rdf,
//     tableName=table_name,
//     mode=SaveMode.Overwrite,
//     postActions=s"Grant all on $table_name} to group pm_users")


In [0]:
from pyspark.sql import *
from pyspark import *
from pyspark.sql import DataFrame

sc = spark._sc
helpers = sc._jvm.com.poshmark.spark.helpers
Config = helpers.Config
Redshift = helpers.Redshift
Config.reloadConfigs(sc._jsc.sc())
Config.registerUDFs()

query = "select * from analytics_scratch.hp_feed_top_level_engagement"
DataFrame(Redshift.getQueryDF(query), spark).createOrReplaceTempView("hp_feed_top_level_engagement")
# query = "select * from athena_scratch.core__users_active_user_segments_daily"
# DataFrame(Redshift.getQueryDF(query), spark).createOrReplaceTempView("core__users_active_user_segments_daily")

In [0]:
%sql

create or replace temporary view vs_hp_feed_top_level_engagement as
with fw as (
select 
  event_date, 
  user_id, 
  daily_au_segment, 
  sum(feed_views) as feed_views
from hp_feed_top_level_engagement 
where event_date between date('2025-10-19') and ('2025-10-25') --10-25
group by 1,2,3
) 
, oi as (
select 
  date(date_trunc('day',booked_at_time)) as booked_date, 
  buyer_id, 
  count(distinct  (listing_id ||order_id )) ordr_cnt,
  COALESCE(SUM(( order_item_gmv * .01  )), 0) AS oi_gmv
from s3_analytics.dw_order_items 
where date(booked_at_time)  between date('2025-10-19') and ('2025-10-25')
group by 1,2

)

select * 
from fw 
left join oi on fw.user_id = oi.buyer_id and fw.event_date = oi.booked_date


In [0]:
%sql
select 
  event_date,
  count(distinct user_id) as dau, 
  sum(feed_views) as feed_views,
  count(distinct case when feed_views >0 then user_id end) as feed_viewers, 
  sum(ordr_cnt) as ordr_cnt,
  sum(case when feed_views >0 then ordr_cnt end) as feed_viewer_ordr


from vs_hp_feed_top_level_engagement 
group by 1 
order by 1 


In [0]:
 %sql
 select
  event_date, 
  daily_au_segment, 
  sum(feed_views) as feed_views, 
  count(distinct case when feed_views>0 then user_id end) as feed_viewer_cnt
from hp_feed_top_level_engagement 
where event_date between date('2025-10-19') and ('2025-10-25') --10-25
group by 1,2
limit 5

In [0]:
%sql
describe hp_feed_top_level_engagement

In [0]:
%sql
select 
  date(date_trunc('day','2025-10-19')) as booked_date, 
  buyer_id, 
  count(distinct  (listing_id ||order_id )) ordr_cnt,
  COALESCE(SUM(( order_item_gmv * .01  )), 0) AS oi_gmv
from s3_analytics.dw_order_items 
where booked_at_time  between date_trunc('day','2025-10-19')   and  date_trunc('day','2025-10-25') 
group by 1,2
limit 5

In [0]:
%sql
SELECT
    dw_order_items.buyer_id  AS "dw_order_items.buyer_id",
        (DATE(dw_order_items.booked_at_time )) AS "dw_order_items.booked_date",
    COUNT(DISTINCT ( dw_order_items.listing_id || dw_order_items.order_id ) ) AS "dw_order_items.count_order_items",
    COALESCE(SUM(( dw_order_items.order_item_gmv * .01  ) ), 0) AS "dw_order_items.total_order_item_gmv"
FROM analytics.dw_order_items  AS dw_order_items
LEFT JOIN analytics.dw_orders  AS dw_orders ON dw_order_items.order_id  = dw_orders.order_id
WHERE ((( dw_order_items.booked_at_time  ) >= (TIMESTAMP '2025-10-17') AND ( dw_order_items.booked_at_time  ) < (TIMESTAMP '2025-10-25'))) AND (dw_orders.is_valid_order = TRUE )
GROUP BY
    1,
    2
ORDER BY
    3 DESC
LIMIT 500

# DAU New Logic

In [0]:
from pyspark.sql import *
from pyspark import *
from pyspark.sql import DataFrame

sc = spark._sc
helpers = sc._jvm.com.poshmark.spark.helpers
Config = helpers.Config
Redshift = helpers.Redshift
Config.reloadConfigs(sc._jsc.sc())
Config.registerUDFs()

query = "select * from analytics_scratch.l365d_shopper_segment"
DataFrame(Redshift.getQueryDF(query), spark).createOrReplaceTempView("l365d_shopper_segment")
query = "select * from athena_scratch.core__users_active_user_segments_daily"
DataFrame(Redshift.getQueryDF(query), spark).createOrReplaceTempView("core__users_active_user_segments_daily")

In [0]:
%scala
val queryDF = spark.sql(s""" 

SELECT
    dw_users.user_id  AS user_id,
        (DATE(dw_user_events_daily.event_date )) AS event_date,

    COUNT(DISTINCT CASE WHEN ( dw_user_events_daily.is_active  ) AND ( dw_user_events_daily.is_valid_user ) THEN dw_user_events_daily.user_id  ELSE NULL END) AS is_active_user
FROM external_scratch.dw_user_events_daily  AS dw_user_events_daily

LEFT JOIN s3_analytics.dw_users  AS dw_users ON dw_user_events_daily.user_id  = dw_users.user_id

WHERE (dw_user_events_daily.event_date ) between (DATE(DATE '2025-10-19')) and (DATE(DATE '2025-10-25')) AND ((coalesce(dw_user_events_daily.app, 'unknown')) in ('unknown','iphone','ipad','external','android','web') )
and dw_users.is_valid_user is true and dw_users.home_domain in ('us')
AND (NOT COALESCE((DATEDIFF((CASE WHEN dw_users.user_status = 'restricted' THEN dw_users.status_updated_at ELSE NULL END), 
  COALESCE(dw_users.guest_joined_at, dw_users.joined_at)) + 1) <= 30, FALSE))

GROUP BY 1,2
ORDER BY 1, 2 asc

""")
val s3_path = "s3://data-tmp-poshmark-production/harshal/feed_personalisation/hp_daily_au_raw_oct_week/"
queryDF.write.format("parquet").mode("Overwrite").save(s3_path)

In [0]:
# read beta_users_all_listings for later use
s3_path = "s3://data-tmp-poshmark-production/harshal/feed_personalisation/hp_daily_au_raw_oct_week/"
df = spark.read.format("parquet").load(s3_path)
df.createOrReplaceTempView("hp_daily_au_raw_oct_week")

### final query to pull the shopper intent clickers

In [0]:
%sql
with fw as (
select t1.event_date, t1.user_id, sum(coalesce(t2.feed_shopper_intent_clicks,0)) as feed_shopper_intent_clicks
from hp_daily_au_raw_oct_week t1
left join hp_feed_top_level_engagement t2 on t1.user_id = t2.user_id and t1.event_date = t2.event_date
where t1.is_active_user = 1
group by 1,2
)
   , oi as (
    select
        date(oi.booked_at_time) as booked_date,
        oi.buyer_id,
        count(distinct  (oi.listing_id ||oi.order_id )) ordr_cnt
        --COALESCE(SUM(( order_item_gmv * .01  )), 0) AS oi_gmv
    from s3_analytics.dw_order_items oi

      LEFT JOIN s3_analytics.dw_orders AS O
                        ON O.order_id = OI.order_id

     WHERE TRUE
       AND DATE(OI.booked_at_time) BETWEEN (DATE '2025-10-19') AND (DATE '2025-10-25')
       AND O.origin_domain ILIKE 'us'
       AND (O.is_valid_order = TRUE)

    group by 1,2

)
   , final as (select *
               from fw
                left join oi on fw.user_id = oi.buyer_id and fw.event_date = oi.booked_date)
select
      event_date,
    count(distinct user_id) as dau,
    count(distinct buyer_id) as buyers,
    sum(feed_shopper_intent_clicks) as feed_shopper_intent_clicks,
    count(distinct case when feed_shopper_intent_clicks >0 then user_id end) as feed_shopper_intent_clicker,
    count(distinct case when feed_shopper_intent_clicks >0 then buyer_id end) as feed_buyers,
    count(distinct case when feed_shopper_intent_clicks <=0 then user_id end) as non_feed_shopper_intent_clicker,
    count(distinct case when feed_shopper_intent_clicks <=0 then buyer_id end) as non_feed_buyers,

    sum(ordr_cnt) as ordr_cnt,
    sum(case when feed_shopper_intent_clicks >0 then ordr_cnt end) as feed_shopper_intent_clicker_ordr,
    sum(case when feed_shopper_intent_clicks <=0 then ordr_cnt end) as non_feed_shopper_intent_clicker_ordr



from final
group by 1
order by 1;

In [0]:
#query to find brand page impression 

In [0]:
%scala
val queryDF = spark.sql(s"""  
select date(r.event_date) as event_date, r.actor.id as user_id, count(case when r.verb = 'click' and r.on.name = 'brand' and r.direct_object.name in ('listing','like','add_to_bundle') then r.actor.id else null end) as brand_listing_clicks,
count(case when r.verb = 'view' and r.direct_object.name in ('brand') then r.actor.id else null end) as brand_page_views

from raw_events r
inner join (select distinct user_id from hp_daily_au_raw_oct_week) as t1 on r.actor.id = t1.user_id

where r.event_date between date('2025-10-19') and ('2025-10-25')
and ((r.verb = 'click' and r.on.name = 'brand' and r.direct_object.name in ('listing','like','add_to_bundle'))
or (r.verb = 'view' and r.direct_object.name in ('brand')))
--and r.properties.listing_id is not null

group by 1, 2
order by 1, 2

""")
val s3_path = "s3://data-tmp-poshmark-production/harshal/feed_personalisation/brand_page_engagement/"
queryDF.write.format("parquet").mode("Overwrite").save(s3_path)

In [0]:
%sql
select   
date(r.event_date) as event_date, 
r.actor.id as user_id, 
count(case when r.verb = 'observe') and r.on.name = 'brand' then r.actor.id else null end) as brand_impression, 
count(case when r.verb = 'click' and r.on.name = 'brand' and r.direct_object.name in ('listing','like','add_to_bundle') then r.actor.id else null end) as brand_listing_clicks,
from raw_events r 
where r.event_date = date('2025-10-19') 
 and ((r.verb = 'observe'  and r.on.name = 'brand') or (r.verb = 'click' and r.on.name = 'brand' and r.direct_object.name in ('listing','like','add_to_bundle'))
or (r.verb = 'view' and r.direct_object.name in ('brand')))
group by 1,2